# Comparing Models with openfit

When fitting dose-response data, a key question is whether adding parameters (Hill3P → Hill4P → Hill5P) is statistically justified. This notebook demonstrates model comparison using AICc, Akaike weights, and the F-test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from openfit import Fit, compare_models

## Simulate asymmetric dose-response data

The true model is Hill5P (asymmetric) — so Hill4P and Hill3P are both misspecified. We'll see if the statistics detect this.

In [ ]:
rng = np.random.default_rng(0)
x = np.array([0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0])

# True 5PL: Bottom=1, Top=99, EC50=5, HillSlope=1.2, HillAsym=0.6
def hill5p(x, Bottom, Top, EC50, HillSlope, HillAsym):
    return Bottom + (Top - Bottom) / (1 + (EC50 / x) ** HillSlope) ** HillAsym

y_true = hill5p(x, 1, 99, 5, 1.2, 0.6)
y = y_true * (1 + rng.normal(0, 0.08, size=len(x)))
y = np.clip(y, 0.1, None)
print("Data generated with true 5PL parameters.")
print("x:", x)
print("y:", np.round(y, 2))

## Fit all three models

In [ ]:
r3 = Fit("hill3p", x, y, weights="1/y2").run()
r4 = Fit("hill4p", x, y, weights="1/y2").run()
r5 = Fit("hill5p", x, y, weights="1/y2").run()

print(f"Hill3P  R²={r3.r_squared:.4f}  AICc={r3.aicc:.2f}  RSS={r3.rss:.4f}")
print(f"Hill4P  R²={r4.r_squared:.4f}  AICc={r4.aicc:.2f}  RSS={r4.rss:.4f}")
print(f"Hill5P  R²={r5.r_squared:.4f}  AICc={r5.aicc:.2f}  RSS={r5.rss:.4f}")

## Formal model comparison

In [ ]:
comp = compare_models([r3, r4, r5])
print(comp.summary())

## Interpret AICc and Akaike weights

The model with the lowest AICc is preferred. ΔAICc > 2 = substantial evidence against the worse model. Akaike weight = probability that model *i* is best. Evidence ratio = weight_best / weight_other.

In [ ]:
print("\nModel ranking by AICc:")
for row in comp.rows:
    print(f"  {row.model_id:8s}  AICc={row.aicc:.2f}  ΔAICc={row.delta_aicc:.2f}  "
          f"weight={row.akaike_weight:.3f}  evidence_ratio={row.evidence_ratio:.1f}x")

## F-test for nested models

Hill3P is nested in Hill4P (Bottom fixed = 0 in 3P). Hill4P is nested in Hill5P (HillAsym fixed = 1 in 4P). The F-test tests whether the extra parameter significantly improves the fit.

In [ ]:
# F-test: Hill3P vs Hill4P (nested)
ft_3v4 = comp.f_test(r3, r4)
print(f"F-test Hill3P vs Hill4P: F={ft_3v4.f_statistic:.2f}, p={ft_3v4.p_value:.4f}")
print("→ Adding Bottom (Hill4P) is", "justified" if ft_3v4.p_value < 0.05 else "not justified", "at p<0.05")

# F-test: Hill4P vs Hill5P (nested)
ft_4v5 = comp.f_test(r4, r5)
print(f"\nF-test Hill4P vs Hill5P: F={ft_4v5.f_statistic:.2f}, p={ft_4v5.p_value:.4f}")
print("→ Adding HillAsym (Hill5P) is", "justified" if ft_4v5.p_value < 0.05 else "not justified", "at p<0.05")

## Visualise all three fits

In [ ]:
x_fine = np.logspace(np.log10(x.min()), np.log10(x.max()), 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = {"hill3p": "green", "hill4p": "blue", "hill5p": "red"}
for res, label in [(r3, "Hill3P"), (r4, "Hill4P"), (r5, "Hill5P")]:
    p = res.params
    if label == "Hill3P":
        yc = p["Top"] * x_fine**p["HillSlope"] / (p["EC50"]**p["HillSlope"] + x_fine**p["HillSlope"])
    elif label == "Hill4P":
        yc = p["Bottom"] + (p["Top"] - p["Bottom"]) / (1 + (p["EC50"] / x_fine)**p["HillSlope"])
    else:
        yc = p["Bottom"] + (p["Top"] - p["Bottom"]) / (1 + (p["EC50"] / x_fine)**p["HillSlope"])**p["HillAsym"]
    axes[0].semilogx(x_fine, yc, label=label, linewidth=2)

axes[0].semilogx(x, y, "ko", markersize=6, zorder=5, label="Data")
axes[0].set_xlabel("Concentration")
axes[0].set_ylabel("Response")
axes[0].set_title("Model fits overlay")
axes[0].legend()

# Residuals for each model
offsets = [-0.05, 0, 0.05]
for (res, label), off in zip([(r3, "Hill3P"), (r4, "Hill4P"), (r5, "Hill5P")], offsets):
    axes[1].semilogx(x * (1 + off), res.standardized_residuals, "o", label=label, alpha=0.8)
axes[1].axhline(0, color="gray")
axes[1].axhline(2, color="k", linestyle="--", alpha=0.3)
axes[1].axhline(-2, color="k", linestyle="--", alpha=0.3)
axes[1].set_xlabel("Concentration")
axes[1].set_ylabel("Standardized Residual")
axes[1].set_title("Residuals by model")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/model_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

## Summary

- AICc correctly identifies Hill5P as the best model when the true curve is asymmetric.
- F-test confirms that both extra parameters are statistically justified.
- In practice: start with Hill4P; if residuals show a systematic pattern (curved trend), try Hill5P.
- Never compare models fit with different weight schemes.